# rearrange-as-sequential-layer composite — cx9: Rearrange layer flattens & restores (B, C, H, W) <-> (B, C*H*W) at the AE bottleneck

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `encoder-decoder-symmetric`, `rearrange-as-sequential-layer`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F
from einops.layers.torch import Rearrange

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "rearrange-as-sequential-layer"
DD_ATOM_IDS = ["encoder-decoder-symmetric", "rearrange-as-sequential-layer"]
DD_SUBTOPICS = ["CNN: Encoder-decoder symmetric layout", "Einops: Rearrange as nn.Sequential layer"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

An AE bottleneck has to go from `(B, C, H, W)` (the encoder's last feature map) to `(B, latent_dim)` (a flat code), and back. The boilerplate way is two `view()` / `flatten()` calls in `forward`. The CLEANER way — and what ARENA uses — is to put `einops.layers.torch.Rearrange` patterns inside the `Sequential` stacks themselves:

- Encoder ends with `Rearrange('b c h w -> b (c h w)')` then `Linear(C*H*W, latent_dim)`.
- Decoder starts with `Linear(latent_dim, C*H*W)` then `Rearrange('b (c h w) -> b c h w', c=C, h=H, w=W)`.

Both directions live inside the `Sequential` body — `forward` is just `self.decoder(self.encoder(x))`. No explicit reshape calls.

**Why both atoms together.** Without the Rearrange-as-layer trick you'd need a custom `forward()` that interleaves the conv stack with shape gymnastics. With it, the encoder/decoder become PURE `nn.Sequential` modules — and the symmetric structure is visible in the source code (every Rearrange in the encoder is mirrored by an inverse Rearrange in the decoder).

**Anatomy.**
```python
encoder = nn.Sequential(
    nn.Conv2d(1, 32, 3, padding=1, stride=2),    # 8 -> 4.
    nn.ReLU(),
    Rearrange('b c h w -> b (c h w)'),           # rearrange-as-sequential-layer.
    nn.Linear(32 * 4 * 4, latent_dim),
)
decoder = nn.Sequential(
    nn.Linear(latent_dim, 32 * 4 * 4),
    Rearrange('b (c h w) -> b c h w', c=32, h=4, w=4),
    nn.Upsample(scale_factor=2),                 # 4 -> 8.
    nn.Conv2d(32, 1, 3, padding=1),
)
```

### Composite Exercise — Rearrange layer flattens & restores (B, C, H, W) <-> (B, C*H*W) at the AE bottleneck

**Atoms exercised together**: `encoder-decoder-symmetric`, `rearrange-as-sequential-layer`

Implement `cx9_make_rearrange_ae(latent_dim)` — return an `nn.Sequential` that round-trips `(B, 1, 8, 8)` -> `(B, 1, 8, 8)`, with a Linear bottleneck of size `latent_dim` in the middle. The WHOLE model must be a single `nn.Sequential` (no custom Module subclass).

Required layer order (call `nn.Sequential(*layers)`):
1. `nn.Conv2d(1, 32, kernel_size=3, padding=1, stride=2)`  # (B, 1, 8, 8) -> (B, 32, 4, 4).
2. `nn.ReLU()`
3. `Rearrange('b c h w -> b (c h w)')`  # (B, 32, 4, 4) -> (B, 32*4*4).
4. `nn.Linear(32 * 4 * 4, latent_dim)`  # bottleneck.
5. `nn.Linear(latent_dim, 32 * 4 * 4)`  # un-bottleneck.
6. `Rearrange('b (c h w) -> b c h w', c=32, h=4, w=4)`  # (B, 32*4*4) -> (B, 32, 4, 4).
7. `nn.Upsample(scale_factor=2)`  # (B, 32, 4, 4) -> (B, 32, 8, 8).
8. `nn.Conv2d(32, 1, kernel_size=3, padding=1)`  # (B, 32, 8, 8) -> (B, 1, 8, 8).

Return the `nn.Sequential` instance.

Test checks:
- Return value is `nn.Sequential` (not a custom Module).
- The 3rd layer is a `Rearrange` Module (capital R — the layer form).
- The 6th layer is also a `Rearrange` (the inverse).
- Round-trip shape: `model(x).shape == (B, 1, 8, 8)` for any batch size.
- Latent shape inside the model: peek at intermediate by running the first 4 layers; result must be `(B, latent_dim)`.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx9_make_rearrange_ae(latent_dim: int) -> 'nn.Sequential':
    """Return an nn.Sequential AE that uses Rearrange layers at the bottleneck."""
    raise NotImplementedError

def _test_cx9():
    model = cx9_make_rearrange_ae(latent_dim=6)
    assert isinstance(model, nn.Sequential), f'expected nn.Sequential, got {type(model).__name__}'

    layers = list(model.children())
    assert len(layers) == 8, f'expected 8 layers; got {len(layers)}: {[type(l).__name__ for l in layers]}'

    # Case A: layers 3 and 6 are Rearrange.
    assert isinstance(layers[2], Rearrange), f'layer 3 must be Rearrange, got {type(layers[2]).__name__}'
    assert isinstance(layers[5], Rearrange), f'layer 6 must be Rearrange, got {type(layers[5]).__name__}'

    # Case B: end-to-end shape parity (B, 1, 8, 8) -> (B, 1, 8, 8).
    for B in (1, 4, 7):
        x = t.randn(B, 1, 8, 8)
        out = model(x)
        assert out.shape == x.shape, f'shape parity broken at B={B}: {tuple(out.shape)}'

    # Case C: intermediate latent shape is (B, latent_dim).
    encoder_part = nn.Sequential(*layers[:4])
    x = t.randn(3, 1, 8, 8)
    z = encoder_part(x)
    assert z.shape == (3, 6), f'latent shape after first 4 layers should be (3, 6); got {tuple(z.shape)}'

    # Case D: the second Rearrange is the INVERSE — its output is (B, 32, 4, 4).
    first_six = nn.Sequential(*layers[:6])
    feat = first_six(x)
    assert feat.shape == (3, 32, 4, 4), (
        f'after the inverse Rearrange, shape should be (3, 32, 4, 4); got {tuple(feat.shape)}'
    )

    # Case E: the Rearrange flatten preserves data (round-trip is lossless when nothing else acts).
    # Build a tiny throwaway Sequential of just the two Rearrange layers + identity linears.
    test_rt = nn.Sequential(
        Rearrange('b c h w -> b (c h w)'),
        Rearrange('b (c h w) -> b c h w', c=32, h=4, w=4),
    )
    feat_in = t.randn(2, 32, 4, 4)
    feat_out = test_rt(feat_in)
    assert t.allclose(feat_in, feat_out, atol=1e-7), 'Rearrange round-trip should be lossless'
    _dd_passed.add('cx9')

_test_cx9()

<details><summary>Show solution — cx9</summary>

```python
def cx9_make_rearrange_ae(latent_dim: int):
    # Atom B (rearrange-as-sequential-layer): Rearrange goes INSIDE Sequential so
    # we never have to write a custom forward(). Atom A (encoder-decoder-symmetric):
    # one conv-stride-2 down, one upsample up; one Rearrange flat, one inverse.
    return nn.Sequential(
        nn.Conv2d(1, 32, kernel_size=3, padding=1, stride=2),
        nn.ReLU(),
        Rearrange('b c h w -> b (c h w)'),
        nn.Linear(32 * 4 * 4, latent_dim),
        nn.Linear(latent_dim, 32 * 4 * 4),
        Rearrange('b (c h w) -> b c h w', c=32, h=4, w=4),
        nn.Upsample(scale_factor=2),
        nn.Conv2d(32, 1, kernel_size=3, padding=1),
    )
```

Two things to notice. First, the SECOND Rearrange needs the explicit `c=32, h=4, w=4` kwargs — einops can't infer them from the flat shape alone. Second, the model is just an `nn.Sequential` — no custom `forward()`, no Module subclass. The whole architecture is declarative.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx9'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx9',
        'subtopics': ["CNN: Encoder-decoder symmetric layout", "Einops: Rearrange as nn.Sequential layer"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()